# ALICE RL — Pure TRL GRPO Training (Colab)

Train any of the 5 ALICE benchmark models using **TRL's GRPO** trainer against the
ALICE adversarial RL environment hosted on Hugging Face Spaces.

**Features demonstrated:**
- ✅ Chain-of-Thought (CoT) prompting
- ✅ Three-tier verifier stack (T1 sandbox → T2 LLM judge → T3 regression)
- ✅ Curriculum manager with discrimination zone auto-escalation
- ✅ Failure bank with novelty scoring
- ✅ Live leaderboard updates via `/leaderboard/update`
- ✅ GRPO group-normalised advantage estimation

**Models:** Qwen2.5-0.5B | Qwen2.5-1.5B | Qwen2.5-3B | SmolLM2-1.7B | Gemma-3-1B

In [ ]:
# Install dependencies
!pip install -q httpx torch transformers peft accelerate trl bitsandbytes

In [ ]:
import os

# ── Configuration ────────────────────────────────────────────────────
ALICE_ENV_URL  = "https://rohanjain1648-alice-rl-environment.hf.space"  # HF Space URL
MODEL_ID       = "Qwen/Qwen2.5-1.5B-Instruct"   # Change to any benchmark model
EPISODES       = 50          # Reduce for quick demo; use 200+ for real training
GROUP_SIZE     = 4           # GRPO group size (actions sampled per step)
MAX_TURNS      = 3
LR             = 1e-5
KL_COEF        = 0.04
MAX_NEW_TOKENS = 128
LOAD_IN_4BIT   = True        # Recommended for T4 Colab GPU

os.environ["ALICE_ENV_URL"] = ALICE_ENV_URL
print(f"Environment: {ALICE_ENV_URL}")
print(f"Model:       {MODEL_ID}")

In [ ]:
# Verify the ALICE environment is reachable
import httpx, json

try:
    health = httpx.get(f"{ALICE_ENV_URL}/health", timeout=15.0).json()
    print("✅ Environment healthy:")
    print(json.dumps(health, indent=2))
except Exception as e:
    print(f"❌ Cannot reach environment: {e}")
    print("Make sure ALICE_ENV_URL is set to a running ALICE Space.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
) if LOAD_IN_4BIT else None

print(f"Loading model (4-bit={LOAD_IN_4BIT})...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print("✅ Model ready")

In [ ]:
def env_reset():
    return httpx.post(f"{ALICE_ENV_URL}/reset", timeout=30.0).json()

def env_step(episode_id, action):
    return httpx.post(f"{ALICE_ENV_URL}/step",
                      json={"episode_id": episode_id, "action": action},
                      timeout=30.0).json()

def sample_response(prompt):
    device = next(model.parameters()).device
    # Chain-of-thought prefix: model reasons before giving final answer
    cot_prompt = f"<task>{prompt}</task>\nThink step by step, then give your final answer.\n<reasoning>"
    inputs = tokenizer(cot_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True, temperature=0.8, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

def collect_rollouts():
    prompts, responses, rewards = [], [], []
    for _ in range(GROUP_SIZE):
        try:
            ep    = env_reset()
            ep_id = ep["episode_id"]
            task  = ep["task"]
            total_reward = 0.0
            last_action  = ""
            for _ in range(MAX_TURNS):
                action = sample_response(task)
                result = env_step(ep_id, action)
                total_reward += result["reward"]
                last_action   = action
                if result.get("done"): break
            prompts.append(task)
            responses.append(last_action)
            rewards.append(total_reward)
        except Exception as exc:
            print(f"Rollout error (skipped): {exc}")
    return prompts, responses, rewards

print("✅ Rollout functions defined")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
all_rewards, all_successes = [], []

for ep in range(1, EPISODES + 1):
    prompts, responses, rewards = collect_rollouts()
    if not rewards:
        continue

    # GRPO update — group-normalised advantages
    rewards_t  = torch.tensor(rewards, dtype=torch.float32)
    advantages = (rewards_t - rewards_t.mean()) / (rewards_t.std().clamp(min=1e-6))
    device     = next(model.parameters()).device
    total_loss = torch.tensor(0.0, device=device)

    for adv, prompt, response in zip(advantages, prompts, responses):
        full_text  = prompt + "\n" + response
        inputs     = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=768).to(device)
        labels     = inputs["input_ids"].clone()
        prompt_len = len(tokenizer(prompt, return_tensors="pt")["input_ids"][0])
        labels[0, :prompt_len] = -100
        out        = model(**inputs, labels=labels)
        log_prob   = -out.loss
        kl_pen     = KL_COEF * (log_prob ** 2)
        total_loss = total_loss + (-(adv.to(device) * log_prob) + kl_pen)

    total_loss = total_loss / max(len(rewards), 1)
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    all_rewards.extend(rewards)
    all_successes.extend([1.0 if r > 0.3 else 0.0 for r in rewards])
    avg_r    = sum(all_rewards[-50:])   / max(len(all_rewards[-50:]),   1)
    avg_succ = sum(all_successes[-50:]) / max(len(all_successes[-50:]), 1)

    if ep % 5 == 0:
        print(f"Ep {ep:4d} | loss={float(total_loss):.4f} | avg_reward={avg_r:.4f} | success={avg_succ:.2%}")

    # Push live scores to leaderboard every 10 episodes
    if ep % 10 == 0:
        try:
            httpx.post(f"{ALICE_ENV_URL}/leaderboard/update",
                       json={"model_id": MODEL_ID, "avg_reward": avg_r,
                             "success_rate": avg_succ, "discrimination_coverage": 0.0,
                             "episodes_run": ep}, timeout=5.0)
        except Exception: pass

print("\n✅ Training complete!")
print(f"Final avg_reward={avg_r:.4f} | success={avg_succ:.2%} | episodes={len(all_rewards)}")

In [ ]:
# Save checkpoint
CKPT_PATH = f"/content/{MODEL_ID.replace('/', '_')}_trl_alice"
model.save_pretrained(CKPT_PATH)
tokenizer.save_pretrained(CKPT_PATH)
print(f"Checkpoint saved to {CKPT_PATH}")

# Optional: push to HF Hub
# from huggingface_hub import HfApi
# HfApi().upload_folder(folder_path=CKPT_PATH, repo_id="your-username/alice-trained",
#                       token="hf_...", commit_message="ALICE TRL GRPO checkpoint")